# BAF.60080 팩터 실습 노트북 (W02 · W10 · W12)

**사용법은 하나뿐입니다 — 위에서부터 셀을 하나씩 실행합니다.** 셀 왼쪽의 ▶ 버튼을 누르거나, 셀을 클릭하고 `Shift + Enter`를 누릅니다.

- 0번 셀은 매번 처음에 한 번 실행합니다(파일 내려받기). Colab은 창을 닫으면 파일이 지워지기 때문입니다.
- 자기 과제에 해당하는 절(W02 / W10 / W12)만 실행하면 됩니다.
- 실행 결과는 자동으로 `log_w02.txt` 같은 로그 파일에 쌓입니다. 마지막 셀에서 내려받아 제출합니다.
- 셀 안의 `--start 196307` 같은 옵션은 바꿔서 다시 실행해도 됩니다. 원본이 망가지지 않습니다.

## 0. 준비 — 데이터와 계산 도구 내려받기 (매번 처음 한 번)

In [ ]:
BASE = "https://raw.githubusercontent.com/keerhee/pension-fund-strategy-and-performance/main/Assignment/BAF634_Factor_Data"
for f in ["factor_lab.py","baf_ind30_vw.csv","baf_mom10.csv","baf_size_bm25.csv","baf_mkt_rf.csv","DATA.md"]:
    !wget -q -O {f} "{BASE}/{f}"
!ls -l
print("준비 완료 — 파일 6개가 보이면 다음 셀로 갑니다.")

## W02 실습 — CAPM과 GRS 검정
아래 셀은 과제 문서의 문제 번호와 같은 순서입니다. 출력은 화면에 나오고, 동시에 `log_w02.txt`에 쌓입니다.

In [ ]:
# 문제 1 — 산업 30개의 평균·표준편차·샤프
!python factor_lab.py summary --set ind30 | tee -a log_w02.txt

In [ ]:
# 문제 2 — CAPM 회귀표와 GRS (전체 기간)
!python factor_lab.py grs --set ind30 --model capm | tee -a log_w02.txt

In [ ]:
# 문제 4 — 알파·베타 산점도 (그림은 로그에 들어가지 않으므로 캡처해 답안에 붙인다)
import sys, importlib; sys.argv=['x']
import factor_lab as fl; importlib.reload(fl)
import matplotlib.pyplot as plt
f = fl.load(); R = fl.excess('ind30', f); g = fl.grs(R, f[['MKT_RF']])
plt.figure(figsize=(7,5)); plt.scatter(g['beta'][:,0], g['alpha'])
for n,b,a in zip(R.columns, g['beta'][:,0], g['alpha']): plt.annotate(n,(b,a),fontsize=8)
plt.axhline(0,color='gray',lw=.8); plt.xlabel('beta (market)'); plt.ylabel('alpha (%/month)')
plt.title('CAPM alpha vs beta, 30 industries'); plt.show()

In [ ]:
# 문제 5 — 모멘텀 10분위, 규모×BE/ME 25개
!python factor_lab.py summary --set mom10 | tee -a log_w02.txt
!python factor_lab.py grs --set mom10 --model capm | tee -a log_w02.txt
!python factor_lab.py summary --set sb25 | tee -a log_w02.txt
!python factor_lab.py grs --set sb25 --model capm | tee -a log_w02.txt

In [ ]:
# 문제 6 — 기간 나누기 (세 묶음 × 두 구간). 마지막 3줄(GRS 요약)만 보여 준다
for s in ['ind30','mom10','sb25']:
    for p in ['--end 196306','--start 196307']:
        print(f'===== {s} {p}')
        !python factor_lab.py grs --set {s} --model capm {p} | tee -a log_w02.txt | tail -3

## W10 실습 — 팩터 구성과 다팩터 GRS
문제 2~5는 모두 `--start 196307`을 씁니다.

In [ ]:
# 문제 1 — 팩터 프리미엄과 상관
!python factor_lab.py factors | tee -a log_w10.txt

In [ ]:
# 문제 2~4 — 세 묶음 × 세 모형 (회귀표 전체가 로그에 남는다)
for s in ['sb25','mom10','ind30']:
    for m in ['capm','ff3','ff3mom']:
        print(f'===== {s} {m}')
        !python factor_lab.py grs --set {s} --model {m} --start 196307 | tee -a log_w10.txt | tail -4

In [ ]:
# 문제 2 — 25개 FF3 회귀표 전체 보기 (어느 포트폴리오가 기각을 이끄는지)
!python factor_lab.py grs --set sb25 --model ff3 --start 196307

In [ ]:
# 문제 6 — 가상 운용사(소형가치 S1B5) 검산
!python factor_lab.py manager --port S1B5 --start 196307 | tee -a log_w10.txt

## W12 팀 프로젝트
### 1부 — 미국 1927~2015 (제공 데이터)
자기 팀 팩터 이름으로 `FACTOR`를 바꿔서 실행합니다 — 1팀 `HML`, 2팀 `UMD`, 3팀 `SMB`. 4팀은 아래 결합 셀을 씁니다.

In [ ]:
FACTOR = "HML"     # ← 1팀 HML · 2팀 UMD · 3팀 SMB
!python factor_lab.py factors | tee -a log_w12.txt
for p in ['--end 196306','--start 196307 --end 198912','--start 199001']:
    print(f'===== {p}')
    !python factor_lab.py factors {p} | tee -a log_w12.txt | head -6
!python factor_lab.py combo --w {FACTOR}=1 | tee -a log_w12.txt
!python factor_lab.py rolling --factor {FACTOR} --window 120 | tee -a log_w12.txt | tail -3

In [ ]:
# 필수 그래프 2장 — 누적수익 곡선과 10년 롤링 프리미엄 (캡처해 덱에 붙인다)
import sys, importlib, numpy as np, pandas as pd, matplotlib.pyplot as plt
sys.argv=['x']; import factor_lab as fl; importlib.reload(fl)
f = fl.load(); x = f[FACTOR].dropna(); idx = pd.PeriodIndex(x.index, freq='M').to_timestamp()
fig, ax = plt.subplots(1, 2, figsize=(13,4))
ax[0].plot(idx, (1+x/100).cumprod().values); ax[0].set_yscale('log'); ax[0].set_title(f'{FACTOR} cumulative (log scale)')
r = x.rolling(120).mean()*12
ax[1].plot(idx, r.values); ax[1].axhline(0,color='gray',lw=.8); ax[1].set_title(f'{FACTOR} 10-year rolling premium (%/yr)')
plt.tight_layout(); plt.show()

In [ ]:
# 4팀 — 결합 세 안 (C안 비중은 문제 2에서 직접 계산해 바꾼다)
!python factor_lab.py factors --end 198912 | tee -a log_w12.txt | head -6
for w in ['HML=1','HML=0.5,UMD=0.5','HML=0.649,UMD=0.351']:
    print(f'===== {w}')
    !python factor_lab.py combo --w {w} | tee -a log_w12.txt
    !python factor_lab.py combo --w {w} --start 199001 | tee -a log_w12.txt | grep COMBO

### 2부 — 표본 밖과 한국 (BAF634 강의 데이터)
1. 브라우저에서 [고금계share 폴더](https://drive.google.com/drive/folders/1vRVfhTRU7YsaT1ndiqTEkxfSEfVIioVZ?usp=sharing)를 연다.
2. 폴더 이름 옆 ▾ → **정리 → 바로가기 추가 → 내 드라이브**를 누른다. (한 번만 하면 된다)
3. 아래 셀을 실행하고, 뜨는 창에서 Google 계정 접근을 허용한다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
src = '/content/drive/MyDrive/고금계share/data'
assert os.path.exists(src), '바로가기가 없습니다 — 위 2번(바로가기 추가)을 먼저 하세요'
!mkdir -p data && ln -sfn "{src}/factors" data/factors
!ls data/factors

In [ ]:
!python factor_lab.py jkp --country usa | tee -a log_w12.txt                  # 부호 확인 + 미국 전체
!python factor_lab.py jkp --country usa --start 201508 | tee -a log_w12.txt   # 문제 4 표본 밖
!python factor_lab.py jkp --country kor | tee -a log_w12.txt                  # 문제 5 한국

## 마지막 — 로그 파일 내려받기 (제출용)

In [ ]:
from google.colab import files
import os
for f in ['log_w02.txt','log_w10.txt','log_w12.txt']:
    if os.path.exists(f): files.download(f)